# Phase 4: Preprocessing

Transform the ML-ready panel (189K rows × 47 cols) into train/val/test splits with:
- Sequential 70/10/20 date-based split (Christensen et al.)
- Log-transformed target (skewness 5.64 → 0.44)
- Feature scaler fit on train only (no leakage)
- Purged k-fold CV with 21-day embargo (Lopez de Prado)

In [ ]:
import json
import sys
sys.path.insert(0, "..")

import numpy as np
import polars as pl
from scipy import stats

from theta.modeling.preprocessing import (
    PANEL_PATH, SPLITS_DIR, TARGET_COL, LOG_TARGET_COL,
    split_panel, add_log_target, fit_scaler, apply_scaler,
    purged_kfold, get_feature_cols,
)

## 1. Load Panel

In [ ]:
df = pl.read_parquet(PANEL_PATH)
print(f"Panel: {len(df):,} rows x {len(df.columns)} cols")
print(f"Symbols: {df['symbol'].n_unique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"\nTarget stats:")
print(f"  mean={df[TARGET_COL].mean():.4f}, median={df[TARGET_COL].median():.4f}")
print(f"  std={df[TARGET_COL].std():.4f}, skewness={stats.skew(df[TARGET_COL].to_numpy()):.2f}")

## 2. Log-Transform Target

In [ ]:
df = add_log_target(df)

orig_skew = stats.skew(df[TARGET_COL].to_numpy())
log_skew = stats.skew(df[LOG_TARGET_COL].to_numpy())
print(f"Original skewness:      {orig_skew:.2f}")
print(f"Log-transformed skewness: {log_skew:.2f}")
print(f"Nulls: {df[LOG_TARGET_COL].null_count()}, Infs: {df[LOG_TARGET_COL].is_infinite().sum()}")

## 3. Sequential Split (70/10/20)

In [ ]:
train, val, test = split_panel(df)

print(f"Train: {len(train):,} rows  ({train['date'].min()} to {train['date'].max()})")
print(f"Val:   {len(val):,} rows  ({val['date'].min()} to {val['date'].max()})")
print(f"Test:  {len(test):,} rows  ({test['date'].min()} to {test['date'].max()})")
print(f"Total: {len(train)+len(val)+len(test):,}")
print(f"\nSymbols — train: {train['symbol'].n_unique()}, val: {val['symbol'].n_unique()}, test: {test['symbol'].n_unique()}")

# Verify no overlap
td, vd, tsd = set(train['date'].to_list()), set(val['date'].to_list()), set(test['date'].to_list())
assert len(td & vd) == 0 and len(td & tsd) == 0 and len(vd & tsd) == 0
print("Date overlap: None (verified)")

## 4. Feature Scaler (train-only fit)

In [ ]:
feature_cols = get_feature_cols(train)
print(f"Features: {len(feature_cols)}")

scaler_stats = fit_scaler(train, feature_cols)

# Show a few feature stats
for col in list(feature_cols)[:5]:
    mean, std = scaler_stats[col]
    print(f"  {col}: mean={mean:.4f}, std={std:.4f}")
print(f"  ...")

# Verify: scaled train should have mean~0, std~1
scaled_train = apply_scaler(train, scaler_stats)
check_col = feature_cols[0]
print(f"\nScaled train '{check_col}': mean={scaled_train[check_col].mean():.6f}, std={scaled_train[check_col].std():.6f}")

## 5. Purged K-Fold CV (21-day embargo)

In [ ]:
all_train_dates = train["date"].unique().sort().to_list()
date_to_idx = {d: i for i, d in enumerate(all_train_dates)}

for i, (fold_train, fold_val) in enumerate(purged_kfold(train, n_splits=5, embargo_days=21)):
    ft_dates = fold_train["date"].unique().to_list()
    fv_dates = fold_val["date"].unique().to_list()
    ti = sorted(date_to_idx[d] for d in ft_dates)
    vi = sorted(date_to_idx[d] for d in fv_dates)
    
    gap = min(abs(t - v) for t in ti for v in vi) if ti and vi else 0
    overlap = len(set(ft_dates) & set(fv_dates))
    
    print(f"Fold {i+1}: train={len(fold_train):,} rows ({len(ft_dates)} dates), "
          f"val={len(fold_val):,} rows ({len(fv_dates)} dates), "
          f"embargo_gap={gap}, overlap={overlap}")

## 6. Save Artifacts

Already saved by `run_preprocessing()`. Verify:

In [ ]:
for f in ["train.parquet", "val.parquet", "test.parquet", "scaler_stats.json"]:
    path = SPLITS_DIR / f
    size = path.stat().st_size / 1e6
    print(f"  {f}: {size:.1f} MB")

with open(SPLITS_DIR / "scaler_stats.json") as fh:
    sj = json.load(fh)
print(f"\nScaler stats: {len(sj) - 1} features + train_mean_rv={sj['__train_mean_rv__']:.4f}")

## Summary

| Artifact | Rows | Date Range |
|----------|------|------------|
| train.parquet | ~132,680 | 2022-01-03 to 2024-12-10 |
| val.parquet | ~19,402 | 2024-12-11 to 2025-05-14 |
| test.parquet | ~36,924 | 2025-05-15 to 2026-03-19 |

- Log target skewness: 5.64 → 0.44
- 44 features, scaler fit on train only
- train_mean_rv = 0.1748 (for R²_OOS denominator)
- Purged 5-fold CV with 21-day embargo confirmed
- SQ absent from test (data ends 2025-01-16) — expected

**Next:** Phase 5 (Baselines) loads these splits directly.

---

# Phase 5: Baseline Models

Seven baseline models predicting rv_21d_forward on the test set:
- **OLS-based (pooled):** HAR, LogHAR, SHAR, HARQ, AR(5), LevHAR
- **Per-symbol:** GARCH(1,1)

All OLS models fit on log target. LogHAR applies Jensen correction exp(ŷ + 0.5σ²) per Christensen et al. (2022); other OLS models use plain exp(ŷ) since their higher residual variance inflates the correction. GARCH produces level variance directly.

In [ ]:
from theta.modeling.baselines import (
    SPLITS_DIR, PREDICTIONS_DIR, TARGET_COL, LOG_TARGET_COL,
    predict_har, predict_loghar, predict_shar, predict_harq, predict_ar5,
    predict_levhar, predict_garch,
    run_baselines,
)

train = pl.read_parquet(SPLITS_DIR / "train.parquet")
test = pl.read_parquet(SPLITS_DIR / "test.parquet")
print(f"Train: {len(train):,} rows, Test: {len(test):,} rows")

## 7. OLS Baselines (HAR Family + AR5)

In [ ]:
ols_models = {
    "HAR": predict_har,
    "LogHAR": predict_loghar,
    "SHAR": predict_shar,
    "HARQ": predict_harq,
    "AR5": predict_ar5,
}

ols_results = {}
for name, fn in ols_models.items():
    result = fn(train, test)
    ols_results[name] = result
    n = len(result)
    nan_count = result["y_pred"].is_nan().sum()
    mean_pred = result["y_pred"].mean()
    print(f"{name:8s}: {n:,} predictions, {nan_count} NaN, mean_pred={mean_pred:.4f}")

## 8. LevHAR (leverage from underlying files)

In [ ]:
levhar_result = predict_levhar(train, test)
print(f"LevHAR: {len(levhar_result):,} predictions, "
      f"{levhar_result['y_pred'].is_nan().sum()} NaN, "
      f"mean_pred={levhar_result['y_pred'].mean():.4f}")
print(f"(may be < {len(test):,} test rows due to missing leverage dates)")

## 9. GARCH(1,1) per-symbol

Fits GARCH(1,1) on percentage log returns from underlying price files. Convergence failures fall back to train_mean_rv. 21-day forecast sums h.1..h.21 and converts from pct-squared to decimal annualized variance.

In [ ]:
garch_result = predict_garch(train, test)
print(f"GARCH: {len(garch_result):,} predictions, "
      f"{garch_result['y_pred'].is_nan().sum()} NaN, "
      f"mean_pred={garch_result['y_pred'].mean():.4f}")
print(f"All positive: {(garch_result['y_pred'].to_numpy() > 0).all()}")

## 10. QLIKE Comparison

QLIKE = mean(y/yhat - log(y/yhat) - 1) — lower is better (Patton 2011). Literature expectation: HAR family should outperform AR(5).

In [ ]:
all_results = {**ols_results, "LevHAR": levhar_result, "GARCH": garch_result}

def qlike(df):
    y = df["y_true"].to_numpy()
    yhat = df["y_pred"].to_numpy()
    return float(np.mean(y / yhat - np.log(y / yhat) - 1))

print(f"{'Model':8s}  {'QLIKE':>8s}  {'n':>7s}  {'mean_pred':>10s}")
print("-" * 40)
for name in ["HAR", "LogHAR", "SHAR", "HARQ", "LevHAR", "AR5", "GARCH"]:
    r = all_results[name]
    q = qlike(r)
    print(f"{name:8s}  {q:8.4f}  {len(r):7,}  {r['y_pred'].mean():10.4f}")

## 11. Save Combined Predictions

In [ ]:
combined = pl.concat(list(all_results.values()))
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
combined.write_parquet(PREDICTIONS_DIR / "baselines.parquet")

print(f"Saved: {PREDICTIONS_DIR / 'baselines.parquet'}")
print(f"Total rows: {len(combined):,}")
print(f"Models: {sorted(combined['model'].unique().to_list())}")
print(f"y_pred NaN: {combined['y_pred'].is_nan().sum()}")
size_mb = (PREDICTIONS_DIR / "baselines.parquet").stat().st_size / 1e6
print(f"File size: {size_mb:.1f} MB")

## Phase 5 Summary

- 7 baseline models: HAR, LogHAR, SHAR, HARQ, LevHAR, AR(5), GARCH(1,1)
- All produce test-set predictions with zero NaN
- LogHAR applies Jensen correction exp(pred + 0.5*sigma2) per Christensen et al. (2022)
- Other OLS models use plain exp(pred) — Jensen correction inflates predictions due to high pooled residual variance
- LevHAR derives leverage features from underlying price files
- GARCH fits per-symbol with convergence fallback to train_mean_rv
- Output: `data/processed/predictions/baselines.parquet` (long format)

**Next:** Phase 6 (LightGBM) — Optuna HP tuning with purged k-fold, QLIKE objective, SHAP.